# Notebook 07 — Robustness Checks

This notebook consolidates all robustness and sensitivity analyses for the thesis.
It reads pre-computed CSV files from `outputs/results/` and reproduces the key
diagnostic figures without re-running the walk-forward pipeline.

**Sections**
1. Refit frequency sensitivity
2. Stress-day threshold sensitivity
3. OOS metrics with bootstrap confidence intervals
4. Signal diagnostics (classifier collapse check)
5. Cross-asset DM test (equity vs non-equity)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Paths ────────────────────────────────────────────────────────────────────
BASE      = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTS   = os.path.join(BASE, 'outputs', 'results')
FIGURES   = os.path.join(BASE, 'outputs', 'figures')

def load(fname):
    path = os.path.join(RESULTS, fname)
    if not os.path.exists(path):
        print(f'  [missing] {path} — run main.py first')
        return None
    return pd.read_csv(path)

print('Base dir:', BASE)
print('Results :', RESULTS)

Base dir: D:\clode9
Results : D:\clode9\outputs\results


---
## 1. Refit Frequency Sensitivity

The walk-forward pipeline refits all ML models every `refit_every` trading days.
The default is **20 days (≈ 1 calendar month)**, matching the standard monthly
rebalancing frequency in institutional portfolio risk management.

This section shows how OOS RMSE changes when we sweep `refit_every` over
`[5, 10, 21, 42, 63]` for the representative pair **BTC-USD vs ^GSPC, w=30**.
A flat curve means the results are insensitive to this design choice.

In [2]:
df_refit = load('refit_sensitivity.csv')

if df_refit is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for model_name, grp in df_refit.groupby('model'):
        axes[0].plot(grp['refit_every'], grp['RMSE'], marker='o', label=model_name)
    axes[0].axvline(20, linestyle='--', color='k', linewidth=1.2, label='Default (20 days)')
    axes[0].set_xlabel('refit_every (trading days)')
    axes[0].set_ylabel('OOS RMSE')
    axes[0].set_title('RMSE vs refit frequency')
    axes[0].legend(fontsize=7, ncol=2)

    for model_name, grp in df_refit.groupby('model'):
        axes[1].plot(grp['refit_every'], grp['R2'], marker='o', label=model_name)
    axes[1].axvline(20, linestyle='--', color='k', linewidth=1.2, label='Default (20 days)')
    axes[1].set_xlabel('refit_every (trading days)')
    axes[1].set_ylabel('OOS R²')
    axes[1].set_title('R² vs refit frequency')
    axes[1].legend(fontsize=7, ncol=2)

    fig.suptitle('Refit Frequency Sensitivity — BTC-USD vs ^GSPC | w=30', fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES, 'refit_sensitivity.png'), dpi=120)
    plt.show()

    print('\nSummary table (RMSE per model × refit_every):')
    pivot = df_refit.pivot_table(index='model', columns='refit_every', values='RMSE')
    display(pivot.round(4))

    # Interpretation
    rmse_range = df_refit.groupby('model')['RMSE'].agg(lambda x: x.max() - x.min())
    print(f'\nRMSE range (max-min) across refit values:')
    print(rmse_range.sort_values(ascending=False).to_string())
    if rmse_range.max() < 0.005:
        print('✓ Curves are flat — results are robust to refit frequency choice.')
    else:
        print('△ Some sensitivity detected — review models with large RMSE range.')


Summary table (RMSE per model × refit_every):


refit_every,5,10,21,42,63
model,,,,,
AR1,0.0648,0.0648,0.0648,0.0647,0.0647
ElasticNet,0.0897,0.0897,0.0897,0.0898,0.0898
GBM,0.0951,0.0975,0.1005,0.1006,0.1018
Naive_Last,0.0651,0.0651,0.0651,0.0651,0.0651
RF,0.0945,0.0957,0.0982,0.0981,0.0979
Ridge,0.0890,0.0890,0.0890,0.0890,0.0890
XGB_GPU,0.0961,0.0990,0.1023,0.1040,0.1053



RMSE range (max-min) across refit values:
model
XGB_GPU       0.009246
GBM           0.006695
RF            0.003775
ElasticNet    0.000084
Ridge         0.000074
AR1           0.000015
Naive_Last    0.000000
△ Some sensitivity detected — review models with large RMSE range.


---
## 2. Stress-Day Threshold Sensitivity

The investor signal layer classifies a day as a **stress event** when the
next-day return falls below `−σ × trailing_volatility`.

The default is **σ = 0.75**, which flags roughly 15–20 % of days.
Lower σ (e.g. 0.5) flags more days (easier to detect but noisier);
higher σ (e.g. 2.0) targets only severe drawdowns (rare, harder to predict).

This sensitivity sweep uses the **already-computed probability predictions**
and just re-evaluates the target label at each σ value — no re-training needed.

In [3]:
df_sigma = load('stress_threshold_sensitivity.csv')

if df_sigma is not None and not df_sigma.empty:
    # Aggregate across all dependency/window combinations
    agg = (df_sigma.groupby(['model', 'stress_sigma'])
                   .agg(F1Down=('F1Down', 'mean'),
                        BalancedAccuracy=('BalancedAccuracy', 'mean'),
                        AUC=('AUC', 'mean'),
                        ExitRate=('ExitRate', 'mean'),
                        n_pos_mean=('n_pos', 'mean'))
                   .reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for model_name, grp in agg.groupby('model'):
        axes[0].plot(grp['stress_sigma'], grp['F1Down'], marker='o', label=model_name)
    axes[0].axvline(0.75, linestyle='--', color='k', linewidth=1.2, label='Default (σ=0.75)')
    axes[0].set_xlabel('stress_sigma (σ threshold)')
    axes[0].set_ylabel('F1 (down-day class)')
    axes[0].set_title('F1 vs stress threshold')
    axes[0].legend(fontsize=8)

    for model_name, grp in agg.groupby('model'):
        axes[1].plot(grp['stress_sigma'], grp['BalancedAccuracy'], marker='o', label=model_name)
    axes[1].axvline(0.75, linestyle='--', color='k', linewidth=1.2, label='Default (σ=0.75)')
    axes[1].set_xlabel('stress_sigma (σ threshold)')
    axes[1].set_ylabel('Balanced Accuracy')
    axes[1].set_title('Balanced Accuracy vs stress threshold')
    axes[1].legend(fontsize=8)

    fig.suptitle('Stress-Day Threshold Sensitivity (averaged across all pairs)', fontsize=12)
    fig.tight_layout()
    plt.show()

    print('\nAveraged metrics by model × stress_sigma:')
    display(agg.pivot_table(index='model', columns='stress_sigma',
                            values='F1Down').round(4))


Averaged metrics by model × stress_sigma:


stress_sigma,0.50,0.75,1.00,1.50,2.00
model,,,,,
GBM_Cls,0.1572,0.1406,0.1271,0.0973,0.0723
Logit,0.2374,0.2054,0.1736,0.1187,0.0705
RF_Cls,0.0317,0.0304,0.0250,0.0226,0.0216


---
## 3. OOS Metrics with Bootstrap Confidence Intervals

To quantify sampling uncertainty in our OOS metric estimates we bootstrap
the test set 1 000 times and compute the empirical 95 % percentile interval.

A wide CI indicates that the RMSE estimate is noisy (e.g. short test period);
non-overlapping CIs between models provide stronger evidence of performance differences.

In [4]:
df_ci = load('metrics_with_ci.csv')

if df_ci is not None and not df_ci.empty:
    # Show one representative pair: BTC-USD vs ^GSPC, w=30
    sub = df_ci[(df_ci['dependency'] == 'corr_BTC-USD_^GSPC') & (df_ci['window'] == 30)].copy()
    if sub.empty:
        sub = df_ci.drop_duplicates(['dependency', 'window']).head(20)
        print('Note: showing first available pair')

    sub = sub.sort_values('rmse_mean').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = range(len(sub))
    ax.barh(y_pos, sub['rmse_mean'], xerr=[
        sub['rmse_mean'] - sub['rmse_ci_lower'],
        sub['rmse_ci_upper'] - sub['rmse_mean']
    ], align='center', alpha=0.75, capsize=4, color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['model'])
    ax.set_xlabel('OOS RMSE (95% bootstrap CI)')
    ax.set_title('OOS RMSE with Bootstrap Confidence Intervals\nBTC-USD vs ^GSPC | w=30')
    fig.tight_layout()
    plt.show()

    print('\nFull CI table:')
    display(sub[['model','rmse_mean','rmse_ci_lower','rmse_ci_upper',
                 'r2_mean','r2_ci_lower','r2_ci_upper','n_test']].round(5))

    # Aggregate across all pairs
    print('\nAggregate average RMSE with CI width (all pairs):')
    agg_ci = (df_ci.groupby('model')
                   .agg(rmse_mean=('rmse_mean','mean'),
                        ci_width=('rmse_ci_upper', lambda x: (x - df_ci.loc[x.index,'rmse_ci_lower']).mean()))
                   .sort_values('rmse_mean'))
    display(agg_ci.round(5))


Full CI table:


,model,rmse_mean,rmse_ci_lower,rmse_ci_upper,r2_mean,r2_ci_lower,r2_ci_upper,n_test
0,AR1,0.06458,0.05961,0.06972,0.94024,0.93033,0.94956,2176
1,Naive_Last,0.06489,0.05990,0.07015,0.93966,0.92969,0.94926,2176
2,Ridge,0.08898,0.08448,0.09345,0.88666,0.87383,0.89823,2176
3,ElasticNet,0.08971,0.08526,0.09419,0.88479,0.87201,0.89632,2176
4,RF,0.09722,0.09228,0.10216,0.86469,0.84989,0.87833,2176
5,GBM,0.10009,0.09516,0.10507,0.85660,0.84120,0.86973,2176
6,XGB_GPU,0.10100,0.09591,0.10615,0.85400,0.83865,0.86786,2176
7,DCC_GARCH,0.19796,0.19207,0.20428,0.43941,0.41205,0.46749,2176



Aggregate average RMSE with CI width (all pairs):


,rmse_mean,ci_width
model,,
AR1,0.06653,0.00964
Naive_Last,0.06718,0.00989
Ridge,0.09140,0.00970
ElasticNet,0.09212,0.00970
RF,0.10094,0.00978
GBM,0.10338,0.01007
XGB_GPU,0.10421,0.00982
DCC_GARCH,0.21459,0.01424


---
## 4. Signal Diagnostics — Classifier Collapse Check

Low F1 scores for the investor signal layer can have two causes:
1. The task is genuinely hard (crypto features do not predict equity stress days)
2. The classifier has **collapsed** — predicting near-constant probabilities,
   never actually firing a signal above the threshold

The diagnostic table below flags any model–pair combination where
the predicted probability standard deviation is below 0.05.

In [5]:
df_diag = load('signal_diagnostics.csv')

if df_diag is not None and not df_diag.empty:
    # Drop duplicate rows (file is appended-to per pair/window)
    df_diag = df_diag.drop_duplicates(subset=['dependency','window','model'])

    print(f'Total classifier instances diagnosed: {len(df_diag)}')
    print(f'Collapsed instances (prob_std < 0.05): {df_diag["collapsed"].sum()}')

    # Summary stats
    summary = (df_diag.groupby('model')
                      .agg(class_balance_mean=('class_balance','mean'),
                           prob_std_mean=('prob_std','mean'),
                           prob_std_min=('prob_std','min'),
                           signal_rate_mean=('signal_rate','mean'),
                           pct_collapsed=('collapsed', lambda x: 100*x.mean()))
                      .round(4))
    print('\nDiagnostic summary by model:')
    display(summary)

    # Highlight collapsed cases
    collapsed = df_diag[df_diag['collapsed']]
    if not collapsed.empty:
        print(f'\n⚠ Collapsed cases ({len(collapsed)}):')
        display(collapsed[['dependency','window','model',
                            'class_balance','prob_std','signal_rate']].round(4))
    else:
        print('\n✓ No collapsed classifiers detected.')

    # Distribution of prob_std across all models
    fig, ax = plt.subplots(figsize=(10, 4))
    for model_name, grp in df_diag.groupby('model'):
        ax.hist(grp['prob_std'], bins=30, alpha=0.5, label=model_name)
    ax.axvline(0.05, linestyle='--', color='red', label='Collapse threshold (0.05)')
    ax.set_xlabel('Predicted probability std dev')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of probability spread across all pair×window combinations')
    ax.legend()
    fig.tight_layout()
    plt.show()

Total classifier instances diagnosed: 3
Collapsed instances (prob_std < 0.05): 0

Diagnostic summary by model:


,class_balance_mean,prob_std_mean,prob_std_min,signal_rate_mean,pct_collapsed
model,,,,,
GBM_Cls,0.1757,0.1676,0.1676,0.1320,0.0
Logit,0.1757,0.0936,0.0936,0.3059,0.0
RF_Cls,0.1757,0.0678,0.0678,0.0381,0.0



✓ No collapsed classifiers detected.


---
## 5. Cross-Asset Performance Difference — DM Test

The thesis covers two groups of traditional assets:
- **Equity**: S&P 500 (`^GSPC`), NASDAQ (`^IXIC`)
- **Non-equity**: Gold (`GLD`), Silver (`SLV`), US Dollar Index (`UUP`)

A natural question is whether the best ML model performs **significantly differently**
for equity vs non-equity pairs.

The Diebold–Mariano test compares the average forecast error across each group.
- DM > 0 means equity errors are larger (non-equity easier to forecast)
- DM < 0 means non-equity errors are larger (equity easier to forecast)
- p < 0.05 means the difference is statistically significant

In [6]:
df_crossdm = load('cross_asset_dm_test.csv')

if df_crossdm is not None and not df_crossdm.empty:
    row = df_crossdm.iloc[0]
    dm_stat = row.get('DM_stat', float('nan'))
    p_val   = row.get('p_value', float('nan'))
    sig     = row.get('significance', '')
    n       = row.get('n', '')

    print('Cross-asset DM test results')
    print('=' * 40)
    print(f"  Group A (equity):     {row.get('assets_a','')}  [{row.get('n_pairs_a','')} pairs]")
    print(f"  Group B (non-equity): {row.get('assets_b','')}  [{row.get('n_pairs_b','')} pairs]")
    print(f"  Window used: w={row.get('window','')}")
    print(f"  DM statistic: {dm_stat:.3f}")
    print(f"  p-value:      {p_val:.4f}  {sig}")
    print(f"  n (obs):      {n}")
    print()

    if pd.notna(p_val):
        if p_val < 0.05:
            direction = 'equity > non-equity' if dm_stat > 0 else 'non-equity > equity'
            print(f'✓ Statistically significant difference ({sig}): {direction} errors')
        else:
            print('○ No statistically significant performance difference between equity and non-equity pairs.')

    display(df_crossdm)

Cross-asset DM test results
  Group A (equity):     ['^GSPC', '^IXIC']  [2 pairs]
  Group B (non-equity): ['GLD', 'SLV', 'UUP']  [3 pairs]
  Window used: w=30
  DM statistic: -9.085
  p-value:      0.0000  ***
  n (obs):      2176

✓ Statistically significant difference (***): non-equity > equity errors


,group_a,assets_a,group_b,assets_b,n_pairs_a,n_pairs_b,window,significance,DM_stat,p_value,n
0,equity,"['^GSPC', '^IXIC']",non_equity,"['GLD', 'SLV', 'UUP']",2,3,30,***,-9.085118,2.225074e-308,2176


---
## Summary

| Check | File | Purpose |
|---|---|---|
| Refit sensitivity | `refit_sensitivity.csv` | Validates choice of refit_every=20 |
| Stress threshold | `stress_threshold_sensitivity.csv` | Validates choice of σ=0.75 |
| Bootstrap CI | `metrics_with_ci.csv` | Quantifies sampling uncertainty in RMSE/R² |
| Signal diagnostics | `signal_diagnostics.csv` | Detects classifier collapse |
| Cross-asset DM | `cross_asset_dm_test.csv` | Tests equity vs non-equity performance gap |